In [2]:
import os
import re
import sys
import uuid
import json
import logging
import shutil
from pathlib import Path
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras import layers, models

import joblib
import optuna

# Ensure 'simss_utils' is in your PYTHONPATH
from simss_utils.JV_steady_state import *
import simss_utils.plot_settings_screen


# ==========================================
# 1. CONFIGURATION & LOGGING
# ==========================================
class Config:
    BASE_DIR = Path.cwd()
    
    # NN / SCLC Paths
    NN_DIR = BASE_DIR / "NN_results" / "20260306-090915_two_defects_7p"
    MODEL_PATH = NN_DIR / "y1" / "20260306-090915_two_defects_7p_y1_trained_model.h5"
    SCALER_PATH = NN_DIR / "scaler_data.json"
    DATASET_NAME = '20260306-090915_two_defects_7p'
    JSON_PATH = NN_DIR / 'simulation_metadata.json'
    EXP_SCLC_PATH = BASE_DIR / 'expData' / '2026-02-05-SE04n3_JVd_#4_2.dat'
    LED_PATH = BASE_DIR / "expData" / 'LED_list.txt'
    
    # SIMSS / JVi Paths
    EXP_JVI_PATH = BASE_DIR / "expData" / "2024-01-04-B10n1 LowLight_JVi_#1_2.dat"
    SESSION_PATH = str(BASE_DIR / "SIMsalabim" / "SimSS")
    DEVICE_PARAMS = str(BASE_DIR / "SIMsalabim" / "SimSS" / "simulation_setup.txt")
    
    # Output Dir
    SCLC_FIT_DIR = NN_DIR / "SE04_2nd_round_fit"
    RES_DIR = SCLC_FIT_DIR / "hybrid_nsga2_d328_Ev"
    RES_DIR.mkdir(parents=True, exist_ok=True)

def setup_logging():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
                        handlers=[logging.FileHandler(Config.RES_DIR / "fit.log"), logging.StreamHandler(sys.stdout)])
    return logging.getLogger(__name__)

logger = setup_logging()

# ==========================================
# 2. UTILITY & PREPROCESSING FUNCTIONS
# ==========================================
def preprocess_for_nn(v_sim, j_sim, target_v_start, target_v_end, target_points=128):
    v_fwd_target = np.logspace(np.log10(target_v_start), np.log10(target_v_end), target_points)
    v_rev_target = -v_fwd_target
    interp_func = interp1d(v_sim, j_sim, kind='linear', fill_value="extrapolate")
    j_combined = np.concatenate([interp_func(v_fwd_target), interp_func(v_rev_target)])
    return np.log(np.clip(np.abs(j_combined), 1e-10, None))

def preprocess_for_fit(v_sim, j_sim, target_v_start, target_v_end, step=0.02):
    v_target = np.arange(target_v_start, target_v_end + step, step)
    interp_func = interp1d(v_sim, j_sim, kind='linear', fill_value="extrapolate")
    return interp_func(v_target)

def generate_trap_file(output_path, E_d_sh, N_t_sh, N_t_d):
    with open(output_path, 'w') as f:
        f.write(f"E\tNtrap\n{E_d_sh}\t{N_t_sh}\n4.8\t{N_t_d}")

def rebuild_decoder_model(nParams=7, Y_shape=256):
    # --- Configuration from your architecture ---
    max_filter = 256
    strides = [2,2,2,2]
    kernel = [2,2,2,2]
    map_size = 16
    
    # --- Architecture Definition ---
    z_in = layers.Input(shape=(nParams,), name="input_layer")
    
    # Initial Dense layers
    z1 = layers.Dense(max_filter)(z_in)
    z1 = tf.keras.activations.swish(z1)
    z1 = layers.Dense(max_filter * map_size)(z1)
    z1 = tf.keras.activations.swish(z1)
    
    # Reshape for Conv1DTranspose
    z1 = layers.Reshape((map_size, max_filter))(z1)
    # Upsampling layers
    z2 = layers.Conv1DTranspose(max_filter//2, kernel[3], strides=strides[3], padding='SAME')(z1)
    z2 = tf.keras.activations.swish(z2)
    z3 = layers.Conv1DTranspose(max_filter//4, kernel[2], strides=strides[2], padding='SAME')(z2)
    z3 = tf.keras.activations.swish(z3)
    z4 = layers.Conv1DTranspose(max_filter//8, kernel[1], strides=strides[1], padding='SAME')(z3)
    z4 = tf.keras.activations.swish(z4)
    z5 = layers.Conv1DTranspose(1, kernel[0], strides=strides[0], padding='SAME')(z4)
    decoded_Y = tf.keras.activations.swish(z5)
    # Final Reshape to match output
    decoded_Y = layers.Reshape((Y_shape,))(decoded_Y)
    
    return models.Model(inputs=z_in, outputs=decoded_Y)

def get_jv_characteristics(voltage, current_density):
    """
    Calculates Jsc, Voc, FF, and MPP from JV data.
    """
    V = np.array(voltage)
    J = np.array(current_density)
    f_jsc = interp1d(V, J, kind='cubic')
    jsc = abs(float(f_jsc(0)))
    f_voc = interp1d(J, V, kind='cubic')
    voc = float(f_voc(0))
    power = V * J
    idx_mpp = np.argmin(power) 
    p_max = abs(power[idx_mpp])

    ff = p_max / (jsc * voc) * 100
    return {
        'Jsc': jsc/10,'Voc': voc,
        'FF': ff,'MPP': p_max/10}        # convert A/m2 to mA/cm2
# ==========================================
# 3. OPTIMIZATION OBJECTIVE
# ==========================================
def objective(trial, regressor, param_scaler, metadata, exp_sclc_log, exp_jvi_interp, v_mask_jvi, j_shunt_jvi, v_mask_sclc):
    ID = str(uuid.uuid4())
    jv_temp = Config.RES_DIR / f"JV_{ID}.dat"
    log_temp = Config.RES_DIR / f"log_{ID}.txt"
    trap_temp = Config.RES_DIR / f"traps_{ID}.txt"

    # --- A. Suggest 10D Parameters ---
    # 1. SCLC Parameters 
    lb_mod = list(metadata['varied_parameters_log'].values())       # log transformed boundary
    
    # *Replace these keys with your actual metadata dictionary keys*
    d_sclc = trial.suggest_float("d_sclc", 324e-9, 332e-9)
    W_L = trial.suggest_float("W_L", lb_mod[1][0], lb_mod[1][1])
    W_R_sclc = trial.suggest_float("W_R_sclc", lb_mod[2][0], lb_mod[2][1])
    mu_e = trial.suggest_float("mu_e", lb_mod[6][0], lb_mod[6][1])
    E_d_sh = trial.suggest_float("E_d_sh", lb_mod[3][0], lb_mod[3][1])
    N_t_sh = trial.suggest_float("N_t_sh", lb_mod[4][0], lb_mod[4][1])
    N_t_d = trial.suggest_float("N_t_d", lb_mod[5][0], lb_mod[5][1])

    # 2. JVi-specific Parameters
    W_R = trial.suggest_float("W_R", 0, 0.3)
    E_v = trial.suggest_float("E_v", 5.3, 5.5)
    G_frac = trial.suggest_float("G_frac", 0.5, 1.0)
    l1_k_dir = trial.suggest_float("l1_k_dir", 1e-18, 1e-14, log=True)
    l1_C_n_bulk = trial.suggest_float("l1_C_n_bulk", 1e-18, 1e-14, log=True)
    l1_mu_p = trial.suggest_float("l1_mu_p", 1e-8, 1e-6, log=True)

    # --- B. Evaluate SCLC (Neural Network) ---
    # Construct input vector for NN (ensure order matches training!)
    j_min, j_max = metadata['j_min'], metadata['j_max']
    nn_input_raw = np.array([[d_sclc, W_L, W_R_sclc, E_d_sh, N_t_sh, N_t_d, mu_e]]) 
    nn_input_scaled = param_scaler.transform(nn_input_raw)
    
    j_sclc_pred = regressor.predict(nn_input_scaled, verbose=0)[0]
    # j_sclc_pred_log = j_sclc_pred*(j_max - j_min) + j_min
    exp_sclc_norm = (exp_sclc_log - j_min) / (j_max - j_min)
    
    # Calculate SCLC Loss (e.g., Mean Absolute Error in Log Space)
    sclc_error = float(np.mean(np.abs(j_sclc_pred[v_mask_sclc] - exp_sclc_norm[v_mask_sclc])))

    # *Crucial optimization:* If SCLC fit is terrible, don't waste time on SIMSS
    if sclc_error > 0.1: # Set this threshold based on your typical NN errors
        raise optuna.TrialPruned()

    # --- C. Evaluate JVi (SIMsalabim) ---
    # Generate temporary trap file for this trial
    generate_trap_file(trap_temp, E_d_sh, np.exp(N_t_sh), np.exp(N_t_d)) # Assuming NN uses log values for traps

    cmd_pars = [
        {'par': 'l1.L', 'val': str(130e-9)},
        {'par': 'W_L', 'val': str(W_L)},
        {'par': 'l1.bulkTrapFile', 'val': str(trap_temp)},
        {'par': 'l1.mu_n', 'val': str(np.exp(mu_e))}, # Convert back from log if necessary
        {'par': 'dev_par_file', 'val': Config.DEVICE_PARAMS},
        {'par': 'JVFile', 'val': str(jv_temp)},
        {'par': 'logFile', 'val': str(log_temp)},
        {'par': 'W_R', 'val': str(E_v-W_R)},
        {'par': 'l1.E_v', 'val': str(E_v)},
        {'par': 'G_frac', 'val': str(G_frac)},
        {'par': 'l1.k_direct', 'val': str(l1_k_dir)},
        {'par': 'l1.C_n_bulk', 'val': str(l1_C_n_bulk)},
        {'par': 'l1.mu_p', 'val': str(l1_mu_p)},
    ]

    try:
        utils_gen.run_simulation('simss', cmd_pars, Config.SESSION_PATH, True, verbose=False)
        data = pd.read_csv(jv_temp, sep=r'\s+')
        
        # Calculate JVi Error (Linear Space MAE)
        max_j = np.max(np.abs(exp_jvi_interp[v_mask_jvi]))
        jvi_error = float(np.mean(np.abs(data['Jext'][v_mask_jvi] + j_shunt_jvi - exp_jvi_interp[v_mask_jvi]))) / max_j
        
    except Exception as e:
        logger.warning(f"SimSS failed: {e}")
        jvi_error = 999.0 # Penalty
    finally:
        # Cleanup temp files
        for f in [jv_temp, log_temp, trap_temp]:
            if f.exists(): f.unlink()

    # Return BOTH errors for Multi-Objective optimization
    return sclc_error, jvi_error


# ==========================================
# PLOTTING AND PREDICTION
# ==========================================
def plot_best_fit_curves(best_params, regressor, param_scaler, metadata, exp_sclc_log,v_target_jvi, exp_jvi_interp, n_trial, v_mask_sclc):
    # 1. Re-run NN prediction for the best params
    nn_input = np.array([[best_params['d_sclc'], best_params['W_L'], best_params['W_R_sclc'], 
                          best_params['E_d_sh'], best_params['N_t_sh'], best_params['N_t_d'], best_params['mu_e']]])
    nn_input_scaled = param_scaler.transform(nn_input)
    j_sclc_pred = regressor.predict(nn_input_scaled, verbose=0)[0]
    j_sclc_pred_log = j_sclc_pred * (metadata['j_max'] - metadata['j_min']) + metadata['j_min']
    
    # save best-fit nn output
    target_v_start, target_v_end = metadata['target_v_start'], metadata['target_v_end']
    combined_v_target = np.tile(np.logspace(np.log10(target_v_start), np.log10(target_v_end), 128), 2)
    data = np.hstack((combined_v_target[v_mask_sclc].reshape(-1,1), np.exp(j_sclc_pred_log[v_mask_sclc]).reshape(-1,1)))
    np.savetxt(Config.RES_DIR / f"SCLC_bestfit{n_trial}.csv", data, header="V, J(A/m2)", delimiter=',')

    # 2. Plot SCLC Comparison
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.semilogy(np.exp(exp_sclc_log[v_mask_sclc]), 'k.', label='Exp SCLC')
    plt.semilogy((np.exp(j_sclc_pred_log[v_mask_sclc])), 'r-', label='NN Fit')
    plt.legend()
    plt.xlabel('Voltage (V)')
    plt.ylabel('Current Density (A/m²)')
    
    jv_temp = Config.RES_DIR / f"JV_bestfit{n_trial}.dat"
    log_temp = Config.RES_DIR / f"log_bestfit{n_trial}.txt"
    trap_temp = Config.RES_DIR / f"traps_bestfit{n_trial}.txt"

    # 3. Plot JVi Comparison (Requires re-running SIMSS once)
    generate_trap_file(trap_temp, best_params['E_d_sh'], np.exp(best_params['N_t_sh']), np.exp(best_params['N_t_d'])) # Assuming NN uses log values for traps

    cmd_pars = [
        {'par': 'l1.L', 'val': str(130e-9)},
        {'par': 'W_L', 'val': str(best_params['W_L'])},
        {'par': 'l1.bulkTrapFile', 'val': str(trap_temp)},
        {'par': 'l1.mu_n', 'val': str(np.exp(best_params['mu_e']))}, # Convert back from log if necessary
        {'par': 'dev_par_file', 'val': Config.DEVICE_PARAMS},
        {'par': 'JVFile', 'val': str(jv_temp)},
        {'par': 'logFile', 'val': str(log_temp)},
        {'par': 'W_R', 'val': str(best_params['E_v']-best_params['W_R'])},
        {'par': 'l1.E_v', 'val': str(best_params['E_v'])},
        {'par': 'G_frac', 'val': str(best_params['G_frac'])},
        {'par': 'l1.k_direct', 'val': str(best_params['l1_k_dir'])},
        {'par': 'l1.C_n_bulk', 'val': str(best_params['l1_C_n_bulk'])},
        {'par': 'l1.mu_p', 'val': str(best_params['l1_mu_p'])},
    ]


    utils_gen.run_simulation('simss', cmd_pars, Config.SESSION_PATH, True, verbose=False)
    if trap_temp.exists(): trap_temp.unlink()
    best_sim_data = pd.read_csv(jv_temp, sep=r'\s+')
    # 2. Visualize comparison
    plt.subplot(1, 2, 2)
    Rpdark= [10.8, 34.6] 
    j_shunt = best_sim_data['Vext']/Rpdark[0] + best_sim_data['Vext']**2/Rpdark[1]
    plt.plot(best_sim_data['Vext'], best_sim_data['Jext'] + j_shunt, 'r-', label='Best Fit (Simulated)', linewidth=2)
    plt.scatter(v_target_jvi, exp_jvi_interp, c='k', label='Experimental Data', alpha=0.5)
    print(get_jv_characteristics(best_sim_data['Vext'], best_sim_data['Jext'] + j_shunt))
    plt.xlabel('Voltage (V)')
    # plt.ylabel('Current Density (A/m²)')
    plt.ylim(min(exp_jvi_interp)*1.2, min(exp_jvi_interp)*(-0.05))
    # plt.legend()
    # plt.show()
    plt.tight_layout()
    plt.savefig(Config.RES_DIR/ f'fit_compare_bestfit{n_trial}.png')
    plt.close()
    
def run_sim_LEDs(sample_id, best_params, save_path, n_trial, rerun=True):
    """
    run simulation over various LED spectrums
    """
    
    LED_list = np.loadtxt(Config.LED_PATH, dtype=str)
    jv_charac = []
    for LED_name in LED_list:
        jv_temp = Config.RES_DIR / f"JV_bestfit{n_trial}_{LED_name}.dat"
        trap_temp = Config.RES_DIR / f"traps_bestfit{n_trial}_{LED_name}.txt"
        
        generate_trap_file(trap_temp, best_params['E_d_sh'], np.exp(best_params['N_t_sh']), np.exp(best_params['N_t_d'])) # Assuming NN uses log values for traps

        cmd_pars = [
            {'par': 'l1.L', 'val': str(130e-9)},
            {'par': 'W_L', 'val': str(best_params['W_L'])},
            {'par':'spectrum','val': f'../Data/LED_{LED_name}_{sample_id}.txt'},
            {'par': 'l1.bulkTrapFile', 'val': str(trap_temp)},
            {'par': 'l1.mu_n', 'val': str(np.exp(best_params['mu_e']))}, # Convert back from log if necessary
            {'par': 'dev_par_file', 'val': Config.DEVICE_PARAMS},
            {'par': 'JVFile', 'val': str(jv_temp)},
            {'par': 'W_R', 'val': str(best_params['E_v']-best_params['W_R'])},
            {'par': 'l1.E_v', 'val': str(best_params['E_v'])},
            {'par': 'G_frac', 'val': str(best_params['G_frac'])},
            {'par': 'l1.k_direct', 'val': str(best_params['l1_k_dir'])},
            {'par': 'l1.C_n_bulk', 'val': str(best_params['l1_C_n_bulk'])},
            {'par': 'l1.mu_p', 'val': str(best_params['l1_mu_p'])},
        ]
        
        if rerun:
            utils_gen.run_simulation('simss', cmd_pars, Config.SESSION_PATH, True, verbose=False)
        
        if trap_temp.exists(): trap_temp.unlink()
        
        data = pd.read_csv(jv_temp,sep=r'\s+')
        j_sim_shunt = data['Vext']/GLOBAL_PARAMS['Rpdark'][0] +  data['Vext']**2/GLOBAL_PARAMS['Rpdark'][1]
        jv_charac.append(get_jv_characteristics(data['Vext'], data['Jext']+j_sim_shunt))
    logger.info(f"best-fit{n_trial} simulation with spectrum from Data/LED_(*)_{sample_id}.txt")
    df_results = pd.DataFrame(jv_charac)
    output_filename = f'JV_charac_{sample_id}_bestfit{n_trial}.csv'
    output_path = os.path.join(save_path, output_filename)
    df_results.to_csv(output_path, index=False)

    return df_results

def plot_phivar(sim_data, sample_id, save_path, n_trial):
    exp_phivar_path = Path('expData')/ 'phivar'/ f'{sample_id}_results.xlsx'
    exp_phivar = pd.read_excel(exp_phivar_path,skiprows=[1,2])
    
    sim_data['phi_flux'] = exp_phivar['phi flux'].values
    
    # Experimental data
    exp_phivar['Jsc_norm'] = exp_phivar['Jsc']* 10 / exp_phivar['phi flux'] 
    exp_phivar['Pout_norm'] = (exp_phivar['Jsc'] * exp_phivar['Voc'] * exp_phivar['FF'])* 10 / exp_phivar['phi flux'] 

    # Simulated 
    sim_data['Jsc_norm'] = sim_data['Jsc']* 10 / sim_data['phi_flux'] 
    sim_data['Pout_norm'] = (sim_data['Jsc'] * sim_data['Voc'] * sim_data['FF'])* 10 / sim_data['phi_flux']

    # 2. Create the (2,2) Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True)
    axes = axes.flatten()

    metrics = [
        ('Jsc_norm', '$\mathit{J}_{sc}$ / $\Phi_{in}$ (As)'),
        ('Voc', '$\mathit{V}_{oc}$ (V)'),
        ('FF', 'FF'),
        ('Pout_norm', '$\mathit{P}_{out}$ / $\mathit{\Phi}_{in}$ (Ws)')
    ]

    for i, (col, label) in enumerate(metrics):
        # Plot experimental data as scatter
        axes[i].scatter(exp_phivar['phi flux'], exp_phivar[col if col in exp_phivar else col], 
                        color='black', label='Exp data', alpha=0.7)
        
        # Plot simulation results as a line
        axes[i].plot(sim_data['phi_flux'], sim_data[col], 
                    color='red', label='Prediction', linewidth=2)
        
        axes[i].set_ylabel(label)
        axes[i].set_xscale('log') # Intensity/Flux is usually best viewed on log scale
        if i == 0:
            axes[i].legend()
        if i in [1, 3]:
            axes[i].yaxis.tick_right()
            axes[i].yaxis.set_label_position('right')

    axes[2].set_xlabel(r'Flux density $\mathit{\Phi}_{in}$ (m$^{-2}$s$^{-1}$)')
    axes[3].set_xlabel(r'Flux density $\mathit{\Phi}_{in}$ (m$^{-2}$s$^{-1}$)')

    plt.tight_layout()
    plt.savefig(save_path/f'phivar_compare_bestfit{n_trial}.png')
    plt.close()
    logger.info(f"Phi vs. jv charac plot saved for {sample_id} at {save_path}")

In [4]:
logger.info("Import best-fit parameters from SCLC fitting...")
# --- Load Initial Parameters from previous step ---
param_file = Config.SCLC_FIT_DIR / 'best_fit_parameters.csv'
df_params = pd.read_csv(param_file)
num_samples = len(list(Config.SCLC_FIT_DIR.glob('fit*.png')))

# Store global parameters for the objective function
params_row = df_params.iloc[0, num_samples:].values
global GLOBAL_PARAMS
GLOBAL_PARAMS = {
    'l1_d_sclc' : 328e-9,     # hard-coded
    'W_L': params_row[0],
    'W_R_sclc' :params_row[1],
    'mu_e': np.log(params_row[5]),
    'E_d_sh': params_row[2],
    'N_t_sh':np.log(params_row[3]),
    'N_t_d': np.log(params_row[4]),
    'Rpdark': [10.8, 34.6]  # Ohm m2, first and second term of dark shunt resistance
}
logger.info(GLOBAL_PARAMS)


logger.info("Initializing Hybrid Optuna Workflow...")

# Load NN Models and Meta
regressor = rebuild_decoder_model(nParams=7, Y_shape=256)
try:
    regressor.load_weights(Config.MODEL_PATH)
except Exception as e:
    logger.info("Error: Weight mismatch. Ensure max_filter and strides match training exactly.")
    print(e)
with open(Config.SCALER_PATH, 'r') as f:
    s_data = json.load(f)
param_scaler = StandardScaler()
param_scaler.mean_ = np.array(s_data["mean"])
param_scaler.scale_ = np.array(s_data["scale"])
# Note: You might need to set n_features_in_ as well
param_scaler.n_features_in_ = len(s_data["mean"])
with open(Config.JSON_PATH, 'r') as f:
    metadata = json.load(f)

# Prepare SCLC Data
exp_sclc_data = np.loadtxt(Config.EXP_SCLC_PATH, comments='#')
exp_sclc_log = preprocess_for_nn(exp_sclc_data[:,0], exp_sclc_data[:,1]*10, 
                                    metadata['target_v_start'], metadata['target_v_end'])
target_v_start, target_v_end = metadata['target_v_start'], metadata['target_v_end']
combined_v_target = np.tile(np.logspace(np.log10(target_v_start), np.log10(target_v_end), 128), 2)
v_mask_sclc = combined_v_target < 3

# Prepare JVi Data
exp_jvi_data = np.loadtxt(Config.EXP_JVI_PATH, comments='#')
target_v_start, target_v_end = 0, 1
exp_jvi_interp = preprocess_for_fit(exp_jvi_data[:,0], exp_jvi_data[:,1]*10*0.16/0.06, target_v_start, target_v_end) # *0.16/0.06 specific for B10 samples

v_target_jvi = np.arange(target_v_start, target_v_end+0.02, 0.02)
v_mask_jvi = exp_jvi_interp < 0
Rpdark = [10.8, 34.6]   # specific for each sample
j_shunt_jvi = v_target_jvi[v_mask_jvi]/Rpdark[0] + v_target_jvi[v_mask_jvi]**2/Rpdark[1]

# Initialize Optuna Study (Minimize BOTH objectives)
study = optuna.create_study(directions=["minimize", "minimize"], 
                            study_name="SCLC_JVi_Hybrid",
                            sampler=optuna.samplers.NSGAIISampler()) 

# --- INJECT STARTING POINT ---
# Here you input the parameters you got from your SCLC-only pre-fit
best_sclc_params = {
    "d_sclc":GLOBAL_PARAMS['l1_d_sclc'],
    "W_L": GLOBAL_PARAMS['W_L'], "W_R_sclc":GLOBAL_PARAMS['W_R_sclc'], "mu_e": GLOBAL_PARAMS['mu_e'], "E_d_sh": GLOBAL_PARAMS['E_d_sh'], 
    "N_t_sh": GLOBAL_PARAMS['N_t_sh'], "N_t_d": GLOBAL_PARAMS['N_t_d'], # Replace with actual bests
    "W_R": 0, "G_frac": 0.57, "l1_k_dir": 2e-17, "l1_C_n_bulk": 1e-18, "l1_mu_p": 3e-7 # Educated guesses
}
study.enqueue_trial(best_sclc_params)
logger.info("Enqueued SCLC pre-fit as starting point.")

# Run Optimization
logger.info("Starting Multi-Objective Optimization...")
# Wrap objective with fixed arguments
study.optimize(lambda t: objective(t, regressor, param_scaler, metadata, exp_sclc_log, exp_jvi_interp, v_mask_jvi, j_shunt_jvi, v_mask_sclc), 
                n_trials=400, 
                n_jobs=1) # Set n_jobs>1 if you want to run SIMSS instances in parallel

# Extract Pareto Front
logger.info("Optimization complete. Best trade-off trials:")
pareto_front = study.best_trials
for t in pareto_front:
    logger.info(f"Trial {t.number}: SCLC Error = {t.values[0]:.4f}, JVi Error = {t.values[1]:.4f}")

# Save Study
df_trials = study.trials_dataframe()
df_trials.to_csv(Config.RES_DIR / 'optuna_history.csv', index=False)

2026-03-19 21:25:03,189 - INFO - Import best-fit parameters from SCLC fitting...
2026-03-19 21:25:03,197 - INFO - {'l1_d_sclc': 3.28e-07, 'W_L': 4.362478392556494, 'W_R_sclc': 4.200000043430672, 'mu_e': -16.354444994345954, 'E_d_sh': 4.228569930482258, 'N_t_sh': 50.656872099270345, 'N_t_d': 48.40830545410283, 'Rpdark': [10.8, 34.6]}
2026-03-19 21:25:03,200 - INFO - Initializing Hybrid Optuna Workflow...


[I 2026-03-19 21:25:03,703] A new study created in memory with name: SCLC_JVi_Hybrid


2026-03-19 21:25:03,705 - INFO - Enqueued SCLC pre-fit as starting point.
2026-03-19 21:25:03,706 - INFO - Starting Multi-Objective Optimization...


[I 2026-03-19 21:25:05,201] Trial 0 finished with values: [0.0015424295905973483, 0.16668103351643057] and parameters: {'d_sclc': 3.28e-07, 'W_L': 4.362478392556494, 'W_R_sclc': 4.200000043430672, 'mu_e': -16.354444994345954, 'E_d_sh': 4.228569930482258, 'N_t_sh': 50.656872099270345, 'N_t_d': 48.40830545410283, 'W_R': 0, 'E_v': 5.462262024503001, 'G_frac': 0.57, 'l1_k_dir': 2e-17, 'l1_C_n_bulk': 1e-18, 'l1_mu_p': 3e-07}.
[I 2026-03-19 21:25:06,411] Trial 1 finished with values: [0.031929087105008636, 0.468331545725394] and parameters: {'d_sclc': 3.271160376931882e-07, 'W_L': 4.269567211880239, 'W_R_sclc': 4.25465151642474, 'mu_e': -15.420603276278884, 'E_d_sh': 4.2586769190555165, 'N_t_sh': 53.09125646523925, 'N_t_d': 48.335276199113295, 'W_R': 0.07100975451049597, 'E_v': 5.392484240366579, 'G_frac': 0.8065465661705988, 'l1_k_dir': 8.118932792858218e-16, 'l1_C_n_bulk': 6.2751299971167014e-15, 'l1_mu_p': 2.5175653219885834e-07}.
[I 2026-03-19 21:25:06,471] Trial 2 pruned. 
[I 2026-03-19

2026-03-19 21:32:27,156 - INFO - Optimization complete. Best trade-off trials:
2026-03-19 21:32:27,242 - INFO - Trial 0: SCLC Error = 0.0015, JVi Error = 0.1667
2026-03-19 21:32:27,243 - INFO - Trial 249: SCLC Error = 0.0058, JVi Error = 0.0504
2026-03-19 21:32:27,245 - INFO - Trial 264: SCLC Error = 0.0015, JVi Error = 0.1667
2026-03-19 21:32:27,246 - INFO - Trial 333: SCLC Error = 0.0107, JVi Error = 0.0168
2026-03-19 21:32:27,247 - INFO - Trial 370: SCLC Error = 0.0096, JVi Error = 0.0394
2026-03-19 21:32:27,249 - INFO - Trial 377: SCLC Error = 0.0018, JVi Error = 0.1080


In [5]:
best_trials = study.best_trials
pareto_data = []
for t in best_trials:
    row = {
        "trial_num": t.number,
        "sclc_error": t.values[0],
        "jvi_error": t.values[1],
        **t.params  
    }
    pareto_data.append(row)
df_best = pd.DataFrame(pareto_data)
df_best.to_csv(Config.RES_DIR / "pareto_front_results.csv", index=False)
sample_id='B10n1-1'
for idx,trial in enumerate(best_trials[1:]):
    best_params = trial.params
    plot_best_fit_curves(best_params, regressor, param_scaler, metadata, exp_sclc_log, v_target_jvi, exp_jvi_interp,
                         idx, v_mask_sclc)
    
    bestfit_phivar = run_sim_LEDs(sample_id, best_params, Config.RES_DIR, idx)
    plot_phivar(bestfit_phivar, sample_id, Config.RES_DIR, idx)

{'Jsc': 11.6861441762, 'Voc': 0.8063085884310464, 'FF': 68.53307683697359, 'MPP': 6.45762402496287}
2026-03-19 21:33:16,054 - INFO - best-fit0 simulation with spectrum from Data/LED_(*)_B10n1-1.txt
2026-03-19 21:33:19,617 - INFO - Phi vs. jv charac plot saved for B10n1-1 at c:\Users\e.kim\Desktop\simS_sclc\NNtraining_simss\NN_results\20260306-090915_two_defects_7p\SE04_2nd_round_fit\hybrid_nsga2_d328_Ev
{'Jsc': 12.7450630493, 'Voc': 0.8707969470498308, 'FF': 74.89607535020538, 'MPP': 8.312237561131571}
2026-03-19 21:33:31,620 - INFO - best-fit1 simulation with spectrum from Data/LED_(*)_B10n1-1.txt
2026-03-19 21:33:33,645 - INFO - Phi vs. jv charac plot saved for B10n1-1 at c:\Users\e.kim\Desktop\simS_sclc\NNtraining_simss\NN_results\20260306-090915_two_defects_7p\SE04_2nd_round_fit\hybrid_nsga2_d328_Ev
{'Jsc': 11.5644833122, 'Voc': 0.8289590261782821, 'FF': 69.39924261088883, 'MPP': 6.652946473389939}
2026-03-19 21:33:47,245 - INFO - best-fit2 simulation with spectrum from Data/LED_(*

In [6]:
fig = optuna.visualization.plot_contour(study, params=["E_d_sh", "N_t_sh"],target=lambda t: t.values[1], # 0 for SCLC, 1 for JVi
    target_name="JVi Error")
fig.show()